# EgyptGPT Autoresearch

Autonomous AI research loop using Claude Code on your Claude subscription.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**Requirements:** Colab GPU runtime, Claude Pro/Max subscription.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf EgyptGPT
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

## 2. Mount Google Drive and prepare data

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%cd /content/EgyptGPT

DATA_DIR = 'data/egypt_char'
DRIVE_DATA = '/content/drive/MyDrive/EgyptGPT_autoresearch/data'
DATA_FILES = ['train.bin', 'val.bin', 'meta.pkl']

# Try loading cached data from Drive
cached = all(os.path.exists(os.path.join(DRIVE_DATA, f)) for f in DATA_FILES)
if cached:
    print('Loading cached data from Google Drive...')
    for f in DATA_FILES:
        shutil.copy(os.path.join(DRIVE_DATA, f), os.path.join(DATA_DIR, f))
else:
    print('No cache. Running prepare.py...')
    # Must run as module import so hiero_transformer submodule is on the path
    !cd /content/EgyptGPT && python -c "from data.egypt_char import prepare"
    # Cache to Drive
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in DATA_FILES:
        shutil.copy(os.path.join(DATA_DIR, f), os.path.join(DRIVE_DATA, f))
    print(f'Cached to {DRIVE_DATA}')

# Verify — fail hard if anything is missing
for f in DATA_FILES:
    path = os.path.join(DATA_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'{path}: {os.path.getsize(path):,} bytes')
print('Data ready.')

## 3. Install Claude Code

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs tmux
!npm install -g @anthropic-ai/claude-code
!claude --version

## 4. Create non-root user and configure everything

`--dangerously-skip-permissions` is blocked under root (Colab default).
This creates a non-root user, sets permissions, configures git + GitHub auth,
checks out the branch, restores `results.tsv`, and configures Claude.

Set `BRANCH` and `CLAUDE_MODEL` below.

In [ ]:
import os, subprocess
from google.colab import userdata

# ── CONFIG ─────────────────────────────────────────────────────────
BRANCH = "autoresearch/run_amber_2"
CLAUDE_MODEL = "claude-opus-4-6"
# ───────────────────────────────────────────────────────────────────

REPO_ROOT = '/content/EgyptGPT'
DRIVE_ROOT = '/content/drive/MyDrive/EgyptGPT_autoresearch'

github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Set GITHUB_TOKEN in Colab Secrets with repo push access.')

subprocess.run(
    ['bash', f'{REPO_ROOT}/scripts/setup_colab.sh'],
    env={**os.environ,
         'GITHUB_TOKEN': github_token,
         'BRANCH': BRANCH,
         'CLAUDE_MODEL': CLAUDE_MODEL,
         'REPO_ROOT': REPO_ROOT,
         'DRIVE_ROOT': DRIVE_ROOT},
    check=True,
)

## 5. Start Drive sync loop

The autoresearch loop in `program.md` instructs Claude to `git reset --hard` whenever an experiment makes things worse. That means `results.tsv` and `RESEARCH_REPORT.md` — the memory of what was tried and what worked — would be wiped along with the bad code changes.

This sync loop copies those files to Google Drive every 150 seconds so the full experiment history survives resets. On the next Colab session (or after a reset), step 4 restores `results.tsv` from Drive so Claude can pick up where it left off without re-running experiments it already tried.

In [ ]:
sync_proc = subprocess.Popen(
    ['bash', f'{REPO_ROOT}/scripts/colab_sync_push.sh'],
    env={**os.environ, 'REPO_ROOT': REPO_ROOT, 'DRIVE_ROOT': DRIVE_ROOT, 'SYNC_SECONDS': '150'},
)
print(f'Sync loop running (pid {sync_proc.pid}), syncing every 150s to {DRIVE_ROOT}')

## 6. Launch autoresearch

Open the **Colab terminal** and run:

```bash
tmux new-session -s autoresearch
bash /content/EgyptGPT/scripts/researcher_shell.sh
```

This runs a CUDA check, `cd`s into the repo, and launches Claude Code with
`--dangerously-skip-permissions` (model is already set by step 4).

> **Need a plain shell instead?** Pass `shell`:
> `bash /content/EgyptGPT/scripts/researcher_shell.sh shell`

### If FRESH run (new branch), paste this prompt:
```
Read program.md in this directory. This is an autoresearch setup.
You are already on the pre-created autoresearch branch printed by the setup cell. Data files are verified.

1. Initialize results.tsv with the header row
2. Begin the experiment loop as described in program.md

Start with the baseline run, then iterate autonomously. Never stop.
```

### If RESUMING a previous run, paste this prompt:
```
Read program.md in this directory. This is an autoresearch setup.
You are already on the pre-created autoresearch branch printed by the setup cell. Data files are verified.

1. Read RESEARCH_REPORT.md for context on what has been tried so far
2. Read results.tsv to see all experiment results
3. Read the current config/train_egypt_char.py, model.py, and train.py to see the current best configuration
4. IMPORTANT: The GPU has changed — re-run the current best config first to establish a new baseline step count, then re-tune lr_decay_iters to match the new step count
5. Continue the experiment loop as described in program.md

Pick up where we left off. Never stop.
```

`results.tsv` is synced to Google Drive every 150 seconds. `GITHUB_TOKEN` is auto-sourced in the researcher shell via `~/.bashrc`, so `git push` works for the experiment loop.

**Detach:** `Ctrl+B` then `D` (keeps it running in background)

**Reattach:** `tmux attach -t autoresearch`

**Stop:** `tmux kill-session -t autoresearch`

## 7. Monitor

Have a look in your Google Drive at [`MyDrive/EgyptGPT_autoresearch/results.tsv`](https://drive.google.com/drive/search?q=title:results.tsv%20type:spreadsheet)

In [ ]:
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'